# Structured Output

Models can be requested to provide theory response in a format matching the given schema .This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

### Pydantic

Pydantic models provide the richest feature set with field validaton , descriptions and nested structures.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")


In [3]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    
    

In [4]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024729880530>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002472A91D3D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'The year the movie was released', 'type': 'integer'}}, 'required': ['title', 'director', 'relea

In [5]:
model.invoke("Tell me about the movie Inception")

AIMessage(content='<think>\nOkay, the user wants me to talk about the movie Inception. Let me start by recalling the basics. It\'s a 2010 film directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio as Dom Cobb. The genre is sci-fi, action, maybe a bit of thriller. The plot revolves around entering people\'s dreams to steal or plant ideas. That\'s the main premise.\n\nI should mention the concept of shared dreaming and the layers of dreams. The idea of planting an idea (inception) versus stealing (extraction). The team he puts together—Arthur, Ariadne, Eames, etc. Each has a specific role. Arthur is the architect, Ariadne the designer, Eames the forger. The use of the totem, like the spinning top, to check if they\'re in a dream. Cobb\'s totem is the top; if it stops spinning, they\'re in reality.\n\nThemes like reality vs. dreams, guilt, Cobb\'s past with his wife Mal. The emotional core is his desire to be with his kids again. The ending where the top keeps spinning

In [6]:
model_with_structured_output.invoke("Tell me about the movie Inception")

Movie(title='Inception', director='Christopher Nolan', release_year=2010)

Value of title can never be a number as its dtype=str

### Message output alongside parsed structure


In [7]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    
model_with_structured_output = model.with_structured_output(Movie,include_raw=True)

model_with_structured_output.invoke("Tell me about the movie Inception")
    
    

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie Inception. I need to figure out if I can use the provided Movie function to get the details. Let me check the function parameters. The function requires title, director, and release_year. I know Inception was directed by Christopher Nolan and came out in 2010. So I should call the Movie function with those parameters. Wait, am I allowed to use this function? The user hasn\'t provided any restrictions, so I\'ll proceed. Let me make sure the parameters are correct. Title is "Inception", director is "Christopher Nolan", release year is 2010. That should cover the required fields. Alright, time to format the tool call correctly.\n', 'tool_calls': [{'id': 'k6s4rwb6v', 'function': {'arguments': '{"director":"Christopher Nolan","release_year":2010,"title":"Inception"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_toke

### Nested Structure

In [9]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    
model_with_structured_output = model.with_structured_output(MovieDetails,include_raw=True)
model_with_structured_output.invoke("Tell me about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie Inception. I need to use the MovieDetails function to get the information. Let me check the parameters required: title, year, cast, and genre. I know the title is "Inception" and the year is 2010. The cast includes Leonardo DiCaprio as Dom Cobb, Joseph Gordon-Levitt as Arthur, and others. The genres are Science Fiction and Action. I should structure the tool call with these details. Let me make sure all required fields are included and formatted correctly as per the function\'s specifications.\n', 'tool_calls': [{'id': 'paq0027bh', 'function': {'arguments': '{"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Ellen Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"}],"genre":["Science Fiction","Action"],"title":"Inception","year":2010}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'tok

### TypedDict


typedict provides a simpler alternative using Python's builtin typing ideal when you dont need runtime validation

In [6]:
from typing_extensions import Annotated,TypedDict

class MovieDict(TypedDict):
    """Movie details as a dictionary"""
    title:Annotated[str, ..., "The title of the movie"]
    year:Annotated[int, ..., "The year the movie was released"]
    cast:Annotated[list[str], ..., "The list of actors in the movie"]
    genre:Annotated[list[str], ..., "The list of genres for the movie"]
    
model_with_structured_output = model.with_structured_output(MovieDict,include_raw=True)
model_with_structured_output.invoke("Tell me about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie Inception. I need to use the MovieDict function to get details. Let me check the parameters required: title, year, cast, genre. I know the title is "Inception" and the year is 2010. The cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, and others. The genres are Science Fiction and Action. I should structure the response with these parameters in the tool_call format.\n', 'tool_calls': [{'id': '3f1chprvp', 'function': {'arguments': '{"cast":["Leonardo DiCaprio","Joseph Gordon-Levitt","Ellen Page","Tom Hardy","Ken Watanabe"],"genre":["Science Fiction","Action"],"title":"Inception","year":2010}', 'name': 'MovieDict'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 177, 'prompt_tokens': 251, 'total_tokens': 428, 'completion_time': 0.301281151, 'completion_tokens_details': {'reasoning_tokens': 100}, 'prompt_time': 0.011412989, 'prom

In [7]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A dataclass is a class typically containing mainly data , although there aren't really any restrictions . We can create it using @dataclass decorator

In [9]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(description="The name of the contact")
    email: str = Field(description="The email address of the contact")
    phone: str = Field(description="The phone number of the contact")
    

agent = create_agent(model, response_format=ContactInfo)

result = agent.invoke({"messages": [{"role": "user", "content": "Get the contact information for John Doe, john@gmail.com, 555-1234."}]})

print(result["structured_response"])

name='John Doe' email='john@gmail.com' phone='555-1234'


In [12]:
from typing_extensions import Annotated, TypedDict
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str
    email: str 
    phone: str 
    

agent = create_agent(model, response_format=ContactInfo)

result = agent.invoke({"messages": [{"role": "user", "content": "Get the contact information for John Doe, john@gmail.com, 555-1234."}]})

print(result["structured_response"])

name='John Doe' email='john@gmail.com' phone='555-1234'


In [14]:
## Dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for an individual"""
    name: str
    email: str 
    phone: str
    
agent = create_agent(model, response_format=ContactInfo)    
result = agent.invoke({"messages": [{"role": "user", "content": "Get the contact information for John Doe, john@gmail.com, 555-1234."}]})

result["structured_response"]

ContactInfo(name='John Doe', email='john@gmail.com', phone='555-1234')